# 🧠 Aula 07: Arquiteturas Convolucionais Profundas para Imagens RGB do Mundo Real

Bem-vindo(a) à **Aula 07** do curso de Redes Neurais Profundas da Graduação da Faculdade Infnet!

Nesta aula, consolidamos a aplicação prática de Visão Computacional com PyTorch em um problema do mundo real:
o diagnóstico automatizado de doenças em lavouras agrícolas através do dataset **PlantVillage**.

Este notebook serve como **base de código e referência direta para o Projeto 2 (Opção CNN)** do curso!

---

### 🎯 Objetivos de Aprendizagem
1. **Ingestão de Imagens RGB Multicanal**: Manipular tensores 4D no formato PyTorch `(batch_size, channels=3, height, width)`.
2. **Pipelines com `torchvision.transforms`**: Aplicar redimensionamento, normalização com médias/desvios padrão do ImageNet e aumento de dados (*Data Augmentation*) no treino.
3. **Convoluções Através dos Canais**: Compreender como múltiplos filtros operam em 3 canais simultaneamente gerando novos mapas de características.
4. **Construção da Arquitetura `PlantVillageCNN`**: Projetar blocos modulares (`nn.Conv2d` -> `nn.BatchNorm2d` -> `nn.ReLU` -> `nn.MaxPool2d`) e calcular a transição exata para `nn.Linear`.
5. **Treinamento e Validação Estável**: Conduzir o ciclo de treinamento com `nn.CrossEntropyLoss`, otimizador `AdamW` e monitoramento da perda e acurácia por época.
6. **Avaliação e Inferência em Campo**: Interpretar matriz de confusão multiclasse e testar predições em folhas individuais.

---
### 1. Instalação de Dependências (Necessário no Google Colab)


In [ ]:
# Executar no Google Colab se as bibliotecas 'datasets' e 'albumentations' não estiverem instaladas
!pip install -q datasets torchvision matplotlib scikit-learn seaborn


---
### 2. Importação de Bibliotecas & Dispositivo


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")


---
### 3. Carregamento do Dataset PlantVillage via Hugging Face
O dataset `mohanty/PlantVillage` contém imagens coloridas RGB de folhas saudáveis e doentes de diversas culturas agrícolas (ex: Tomate, Batata, Maçã).


In [ ]:
# Carregar dataset via Hugging Face Datasets
print("Carregando dataset PlantVillage da Hugging Face...")
raw_dataset = load_dataset("mohanty/PlantVillage", split="train[:5000]") # Subconjunto de 5.000 imagens para treino ágil

# Obter lista de classes e mapeamento de nomes
class_names = raw_dataset.features['label'].names
num_classes = len(class_names)

print(f"Total de amostras carregadas: {len(raw_dataset)}")
print(f"Número de classes agrícolas: {num_classes}")
print("Exemplo de classes:", class_names[:5])


---
### 4. Wrapper PyTorch Dataset & Transformações de Imagem RGB
Para alimentar uma CNN com imagens coloridas no PyTorch, definimos:
- Redimensionamento para $128 \times 128$ pixels.
- Aumento de Dados (*Data Augmentation*) com rotação e espelhamento horizontal.
- Conversão para Tensor e Normalização ImageNet $(\mu = [0.485, 0.456, 0.406], \sigma = [0.229, 0.224, 0.225])$.
- Formato final dos Tensores de Entrada: **$(3, 128, 128)$**


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PlantVillageDatasetWrapper(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
        
    def __len__(self):
        return len(self.dataset)
        
    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert('RGB')  # Garantir 3 canais RGB
        label = item['label']
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Divisão de Treino (80%) e Validação (20%)
train_size = int(0.8 * len(raw_dataset))
val_size = len(raw_dataset) - train_size

train_hf, val_hf = torch.utils.data.random_split(raw_dataset, [train_size, val_size])

train_dataset = PlantVillageDatasetWrapper(train_hf, transform=train_transform)
val_dataset = PlantVillageDatasetWrapper(val_hf, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Amostras de Treino: {len(train_dataset)} | Amostras de Validação: {len(val_dataset)}")


---
### 5. Visualização de Amostras Coloridas (RGB)


In [ ]:
def denormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.numpy().transpose((1, 2, 0))
    img = std * img + mean
    return np.clip(img, 0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Amostras do Dataset PlantVillage (Saúde Foliar)", fontsize=14, fontweight='bold')

for i in range(8):
    row, col = i // 4, i % 4
    axes[row, col].imshow(denormalize(images[i]))
    title_short = class_names[labels[i].item()].split('___')[-1]
    axes[row, col].set_title(f"Classe: {title_short}", fontsize=9)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()


---
### 6. Arquitetura da CNN Profunda (`PlantVillageCNN`)
Para processar imagens RGB de $128 \times 128$, construímos uma CNN com 3 blocos convolucionais e **Batch Normalization** para estabilização de treino.

- **Bloco 1:** `Conv2d(3, 32, 3)` + `BatchNorm` + `ReLU` + `MaxPool2d(2, 2)` $\rightarrow (32, 64, 64)$
- **Bloco 2:** `Conv2d(32, 64, 3)` + `BatchNorm` + `ReLU` + `MaxPool2d(2, 2)` $\rightarrow (64, 32, 32)$
- **Bloco 3:** `Conv2d(64, 128, 3)` + `BatchNorm` + `ReLU` + `MaxPool2d(2, 2)` $\rightarrow (128, 16, 16)$
- **Classificador Denso:** `Flatten()` + `Linear(128 * 16 * 16, 256)` + `Dropout(0.4)` + `Linear(256, num_classes)`


In [ ]:
class PlantVillageCNN(nn.Module):
    def __init__(self, num_classes):
        super(PlantVillageCNN, self).__init__()
        
        self.features = nn.Sequential(
            # Bloco 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloco 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloco 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        logits = self.classifier(x)
        return logits

model_plant = PlantVillageCNN(num_classes=num_classes).to(device)
print(model_plant)

total_params_plant = sum(p.numel() for p in model_plant.parameters() if p.requires_grad)
print(f"Total de parâmetros da CNN PlantVillage: {total_params_plant:,}")


---
### 7. Treinamento & Validação da CNN AgriTech


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_plant.parameters(), lr=0.001)

epochs = 10

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(epochs):
    # Fases de Treino
    model_plant.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model_plant(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / total
    epoch_train_acc = (correct / total) * 100
    
    # Fase de Validação
    model_plant.eval()
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_plant(images)
            loss = criterion(outputs, labels)
            
            val_running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    epoch_val_loss = val_running_loss / val_total
    epoch_val_acc = (val_correct / val_total) * 100
    
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    train_accs.append(epoch_train_acc)
    val_accs.append(epoch_val_acc)
    
    print(f"Época [{epoch+1:02d}/{epochs:02d}] | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")


---
### 8. Gráficos de Desempenho (Curva de Aprendizado)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot da Perda
ax1.plot(range(1, epochs+1), train_losses, label='Treino Loss', marker='o', color='#0A345D')
ax1.plot(range(1, epochs+1), val_losses, label='Validação Loss', marker='s', color='#FF7043')
ax1.set_title("Curva de Perda (Loss)")
ax1.set_xlabel("Época")
ax1.set_ylabel("CrossEntropy Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot da Acurácia
ax2.plot(range(1, epochs+1), train_accs, label='Treino Acc (%)', marker='o', color='#1BB5D8')
ax2.plot(range(1, epochs+1), val_accs, label='Validação Acc (%)', marker='s', color='#7CB342')
ax2.set_title("Curva de Acurácia (%)")
ax2.set_xlabel("Época")
ax2.set_ylabel("Acurácia (%)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
### 9. Função de Inferência em Novas Imagens (Predição Agrícola)


In [ ]:
def predict_leaf(model, image_tensor):
    model.eval()
    with torch.no_grad():
        input_b = image_tensor.unsqueeze(0).to(device)
        output = model(input_b)
        probabilities = torch.softmax(output, dim=1)
        prob, pred_class = torch.max(probabilities, 1)
        
    class_name = class_names[pred_class.item()]
    confidence = prob.item() * 100
    return class_name, confidence

# Testar em 1 amostra de validação
test_img, test_lbl = val_dataset[0]
predicted_label, conf = predict_leaf(model_plant, test_img)
real_label = class_names[test_lbl]

plt.figure(figsize=(5, 5))
plt.imshow(denormalize(test_img))
plt.title(f"Real: {real_label.split('___')[-1]}\nPredito: {predicted_label.split('___')[-1]} ({conf:.1f}%)", color='green' if predicted_label == real_label else 'red')
plt.axis('off')
plt.show()


---
## 🏆 Conclusão e Resumo Pedagógico da Aula 07

Parabéns! Você construiu um pipeline completo de Visão Computacional para imagens do mundo real:
- **Ingestão**: Carregou imagens RGB multicanal tratando variações de iluminação com normalização padrão.
- **Augmentation**: Preveniu sobreajuste ensinando a rede que rotações e espelhamentos preservam o diagnóstico da folha.
- **Arquitetura Profunda**: Combinou convoluções, normalização de ativação 2D (`BatchNorm2d`), pooling e dropout.
- **Projeto 2**: Este código serve como fundação direta para quem escolher a Opção CNN no Projeto 2 da disciplina!

Na **Aula 08**, exploraremos o aprendizado não-supervisionado com **Autoencoders Convolucionais**, espaço latente e detecção de anomalias industriais!